In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)


In [ ]:
import pandas as pd

filename = "ks-projects-201801.csv"
try:
    df = pd.read_csv(filename, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(filename, encoding="latin1")

print(f"Dataset Shape: {df.shape}")
df.head()

##Rebuild the clean binary target

In [ ]:
TARGET = "state"

binary_df = df[df[TARGET].isin(["successful", "failed"])].copy()
binary_df["target"] = (binary_df[TARGET] == "successful").astype(int)

print(f"Rows kept: {len(binary_df):,} ({len(binary_df)/len(df)*100:.1f}% of original)")
binary_df["target"].value_counts(normalize=True).round(3)

## Drop unusable / corrupt rows

In [ ]:
binary_df["launched_dt"] = pd.to_datetime(binary_df["launched"], errors="coerce")
binary_df["deadline_dt"] = pd.to_datetime(binary_df["deadline"], errors="coerce")

before = len(binary_df)

bad_epoch = binary_df["launched_dt"].dt.year <= 1971
bad_order = binary_df["deadline_dt"] < binary_df["launched_dt"]
bad_goal = binary_df["usd_goal_real"] <= 0

binary_df = binary_df[~(bad_epoch | bad_order | bad_goal)].copy()

print(f"Dropped {before - len(binary_df):,} corrupt rows ({(before - len(binary_df))/before*100:.3f}%)")
print(f"Remaining rows: {len(binary_df):,}")


## Date-based features

In [ ]:
binary_df["campaign_duration"] = (
    binary_df["deadline_dt"] - binary_df["launched_dt"]
).dt.days

binary_df["launch_month"] = binary_df["launched_dt"].dt.month
binary_df["launch_weekday"] = binary_df["launched_dt"].dt.day_name()
binary_df["launch_quarter"] = binary_df["launched_dt"].dt.quarter

binary_df[["campaign_duration", "launch_month", "launch_weekday", "launch_quarter"]].describe(include="all")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure the output directory exists
os.makedirs("reports/figures", exist_ok=True)

plt.figure(figsize=(10, 5))
sns.boxplot(data=binary_df, x=TARGET, y="campaign_duration")
plt.title("Campaign Duration vs Outcome")
plt.tight_layout()
plt.savefig("reports/figures/duration_vs_outcome.png", dpi=300, bbox_inches="tight")
plt.show()

## Goal Log

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure the output directory exists
os.makedirs("reports/figures", exist_ok=True)

binary_df["goal_log"] = np.log1p(binary_df["usd_goal_real"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(binary_df["usd_goal_real"], bins=50, ax=axes[0])
axes[0].set_title("Goal (raw)")
sns.histplot(binary_df["goal_log"], bins=50, ax=axes[1], color="orange")
axes[1].set_title("Goal (log1p)")
plt.tight_layout()
plt.savefig("reports/figures/goal_log_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

##Outcome-adjacent features

In [ ]:
binary_df["funding_ratio"] = (
    binary_df["usd_pledged_real"] / binary_df["usd_goal_real"].replace(0, np.nan)
)
binary_df["avg_pledge"] = (
    binary_df["usd_pledged_real"] / binary_df["backers"].replace(0, np.nan)
)

binary_df[["funding_ratio", "avg_pledge"]].describe()


##Category frequency encoding

In [ ]:
category_freq = binary_df["category"].value_counts(normalize=True)
binary_df["category_frequency"] = binary_df["category"].map(category_freq)

binary_df[["category", "category_frequency"]].drop_duplicates().sort_values(
    "category_frequency", ascending=False
).head(10)


## Country success rate

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols_preview = [
    "main_category", "category", "currency", "country", "usd_goal_real",
    "goal_log", "campaign_duration", "launch_month", "launch_weekday",
    "launch_quarter", "category_frequency",
]

train_df, test_df = train_test_split(
    binary_df,
    test_size=0.2,
    stratify=binary_df["target"],
    random_state=RANDOM_STATE,
)

# Compute country success rate on TRAIN ONLY
country_success_rate = train_df.groupby("country")["target"].mean()
global_success_rate = train_df["target"].mean()  # fallback for unseen countries

train_df = train_df.copy()
test_df = test_df.copy()
train_df["country_success_rate"] = train_df["country"].map(country_success_rate)
test_df["country_success_rate"] = (
    test_df["country"].map(country_success_rate).fillna(global_success_rate)
)

print(f"Train rows: {len(train_df):,}  |  Test rows: {len(test_df):,}")
train_df[["country", "country_success_rate"]].drop_duplicates().sort_values(
    "country_success_rate", ascending=False
).head(10)


##Assemble the final pre-launch feature set

In [ ]:
safe_feature_cols = [
    "main_category", "category", "currency", "country",
    "usd_goal_real", "goal_log", "campaign_duration",
    "launch_month", "launch_weekday", "launch_quarter",
    "category_frequency", "country_success_rate",
]

analysis_only_cols = ["funding_ratio", "avg_pledge"]  # keep for SHAP/pattern-mining later, not Day 4 training

X_train = train_df[safe_feature_cols]
y_train = train_df["target"]
X_test = test_df[safe_feature_cols]
y_test = test_df["target"]

print("X_train:", X_train.shape, " X_test:", X_test.shape)
X_train.head()


### Save processed data for Day 4

In [ ]:
train_df.to_csv("data/processed/train_features.csv", index=False)
test_df.to_csv("data/processed/test_features.csv", index=False)

print("Saved:")
print(" - data/processed/train_features.csv", train_df.shape)
print(" - data/processed/test_features.csv ", test_df.shape)